In [1]:
from utils.bias_pipeline import BiasPipeline
pipeline = BiasPipeline()


c:\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
04/16/2026 02:45:31 - INFO - 	 missing_keys: []
04/16/2026 02:45:31 - INFO - 	 unexpected_keys: []
04/16/2026 02:45:31 - INFO - 	 mismatched_keys: []
04/16/2026 02:45:31 - INFO - 	 error_msgs: []
04/16/2026 02:45:31 - INFO - 	 Model Parameters: 90.5M, Transformer: 82.1M, Coref head: 8.4M


In [24]:
#get raw text 
text = pipeline.from_json_basil("DATA/BASIL-main/articles/2019/0d10341a-9dba-4374-a524-814c300d1611_1.json")

In [25]:
#Step one get all subjects
doc = pipeline.NER_nlp(text)

In [26]:
#GET only people
people = [(ent.start_char, ent.end_char, ent.text) for ent in doc.ents if ent.label_ == "PERSON"]
people
        

[(10, 15, 'Trump'),
 (431, 436, 'Trump'),
 (703, 708, 'Trump'),
 (1142, 1147, 'Trump'),
 (1262, 1275, 'Chuck Schumer'),
 (1290, 1295, 'Trump'),
 (1394, 1399, 'Trump'),
 (1901, 1913, 'Nancy Pelosi'),
 (2174, 2189, 'Mitch McConnell'),
 (2203, 2208, 'Trump'),
 (2312, 2317, 'Trump'),
 (2439, 2444, 'Pence'),
 (2457, 2470, 'Jared Kushner'),
 (2579, 2584, 'Trump'),
 (2674, 2679, 'Trump'),
 (2839, 2844, 'Trump'),
 (2928, 2933, 'Trump'),
 (2986, 2992, 'Pelosi'),
 (4036, 4042, 'Pelosi'),
 (4218, 4224, 'Pelosi'),
 (4401, 4410, 'McConnell'),
 (4701, 4714, 'Sarah Sanders'),
 (4777, 4783, 'Pelosi'),
 (4919, 4931, 'Nancy Pelosi'),
 (5194, 5199, 'Pence')]

In [62]:
data = {}
for linked_ent in doc._.linkedEntities:
    #using linkedEntities as a 
    span = linked_ent.get_span()
    #Bool if PERSON
    is_person = any(start <= span.start_char and span.end_char <= end for start, end, _ in people)
    
    # if linked_ent.description == "family name":
        #reloop to add to subject
        # continue
    if is_person and linked_ent.description != "family name":
        id = linked_ent.get_id()
        if id not in data:
            data[linked_ent.get_id()] ={
                "name": linked_ent.get_label(),
                "span": [(span.start_char, span.end_char)],
                "description": linked_ent.description,
                
        }

data 

{22686: {'name': 'Donald Trump',
  'span': [(431, 436)],
  'description': '45th and current president of the United States'},
 380900: {'name': 'Charles Ellis Schumer',
  'span': [(1262, 1275)],
  'description': 'United States Senator from New York'},
 170581: {'name': 'Nancy Pelosi',
  'span': [(1901, 1913)],
  'description': 'Speaker of the United States House of Representatives'},
 355522: {'name': 'Mitch McConnell',
  'span': [(2174, 2189)],
  'description': 'United States Senator from Kentucky'},
 13628723: {'name': 'Jared Kushner',
  'span': [(2457, 2470)],
  'description': 'American investor, real-estate developer, newspaper publisher, and senior advisor to President Donald Trump'},
 27986907: {'name': 'Sarah Sanders',
  'span': [(4701, 4714)],
  'description': 'American political press secretary'},
 55594613: {'name': 'Matt Pence',
  'span': [(5194, 5199)],
  'description': 'American musician, producer, and drummer'}}

In [39]:
preds = pipeline.Coref_model.predict(text)

04/16/2026 03:05:17 - INFO - 	 Tokenize 1 inputs...
Map: 100%|██████████| 1/1 [00:00<00:00, 21.21 examples/s]
04/16/2026 03:05:17 - INFO - 	 ***** Running Inference on 1 texts *****
Inference: 100%|██████████| 1/1 [00:00<00:00,  1.91it/s]


In [61]:
spans = preds.get_clusters(as_strings=False)
spans[-2]

[(2424, 2444), (5194, 5199)]

In [60]:
clusters = preds.get_clusters()
clusters[-2]

['Vice President Pence', 'Pence']